# 신용카드 이탈 예측 실무 분석 보고서

## 분석 개요
본 노트북은 제공된 파이썬 코드를 기반으로 **신용카드 고객 이탈(churn) 예측 모델을 구축하고 성능을 검증**하는 실무형 분석 보고서입니다.

## 보고서 활용 목적
- 데이터 사이언스 팀: 모델링 재현 및 성능 검증
- 서비스 기획/운영 팀: 이탈 위험 고객 선별 로직 이해
- 배포 담당자: 추론 파이프라인 반영 전 주의사항 검토

## 분석 포인트
- 현재 데이터는 **이탈 고객 비율이 낮은 불균형 데이터**이므로 정확도만으로 성능을 판단하면 왜곡될 수 있습니다.
- 따라서 본 보고서에서는 **ROC-AUC, PR-AUC, Recall, Precision, F1-score**를 함께 확인합니다.
- 하이퍼파라미터 탐색은 단순 최고 점수 확인이 아니라 **과적합 여부와 운영 안정성**까지 고려하는 방향으로 해석합니다.

## 1. 분석 환경 설정

### 비즈니스 목적과 기대 효과
분석에 필요한 라이브러리와 출력 포맷을 먼저 표준화하면, 팀원 간 실행 결과 차이를 줄이고 재현 가능한 분석 환경을 확보할 수 있습니다.

### 분석 포인트
- 데이터프레임은 `display()` 기반으로 출력해 가독성을 높입니다.
- 시각화 제목, 축 이름, 범례를 명확히 지정해 보고서 전달력을 높입니다.
- 유지보수 시 폰트, 경로, 네트워크 호출 실패에 유의해야 합니다.

In [ ]:

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests

from IPython.display import display

# 보고서 가독성을 위해 pandas 출력 옵션을 조정합니다.
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

# 시각화 기본 크기를 지정해 셀 실행 시 그래프가 지나치게 작아지지 않도록 합니다.
plt.rcParams["figure.figsize"] = (10, 6)

## 2. 데이터 적재 함수 정의

### 비즈니스 목적과 기대 효과
분석에서 가장 먼저 필요한 것은 **데이터를 안정적으로 불러오는 표준 함수**입니다.  
데이터 적재 로직을 함수화하면, 이후 모델링·배포 단계에서도 동일한 원천 데이터를 일관되게 사용할 수 있습니다.

### 분석 포인트
- API 기반 데이터 로드는 로컬 파일 경로 의존성을 줄이는 장점이 있습니다.
- 반면 네트워크 장애, 응답 포맷 변경, 상태 코드 실패에 취약하므로 방어 로직이 필요합니다.
- 실무 배포 시에는 응답 검증과 예외 처리를 반드시 추가해야 합니다.

In [ ]:

def fetch_creditcard(X_y_split: bool = False) -> pd.DataFrame | tuple[pd.DataFrame, pd.Series]:
    # API 응답 실패 시 후속 셀 전체가 잘못된 데이터로 실행되는 것을 방지하기 위해 상태 코드를 검증합니다.
    response = requests.get(
        "http://pipeline:8000/dataset/creditcard-churn",
        params={"X_y_split": X_y_split},
        timeout=30
    )
    response.raise_for_status()

    payload = response.json()

    # 실무 배포 시 응답 스키마 변경 가능성을 고려해야 합니다.
    if X_y_split:
        X = pd.DataFrame(payload["x"], index=payload["index"])
        y = pd.Series(payload["y"], index=payload["index"], name="churn")
        return X, y

    df = pd.DataFrame(payload["data"], index=payload["index"])
    return df

## 3. 원본 데이터 로드

### 비즈니스 목적과 기대 효과
원본 데이터를 불러온 뒤 컬럼 구조와 샘플 행을 빠르게 확인하면, 이후 전처리 및 모델링 방향을 더 정확히 설정할 수 있습니다.

### 분석 포인트
- 식별자 컬럼은 예측에 직접적인 의미가 없고 누수(leakage)를 유발할 수 있으므로 제거 후보입니다.
- 샘플 데이터를 먼저 보면 범주형 인코딩 여부, 수치형 스케일 범위, 결측 가능성을 빠르게 점검할 수 있습니다.

In [ ]:

bank_df = fetch_creditcard()

# 고객 고유 ID는 예측이 아니라 식별 목적이므로 모델 입력에서 제거합니다.
if "creditcard_churn_id" in bank_df.columns:
    bank_df = bank_df.drop("creditcard_churn_id", axis=1)

display(bank_df.head())

## 4. 데이터 구조 및 컬럼 점검

### 비즈니스 목적과 기대 효과
학습 전 컬럼 목록과 데이터 타입을 점검하면, 범주형/수치형 변수 처리 전략과 모델 입력 가능 여부를 빠르게 판단할 수 있습니다.

### 분석 포인트
- 일부 컬럼은 숫자형처럼 보여도 실제로는 범주형 코드일 수 있습니다.
- 배포 시 학습 데이터와 추론 데이터의 타입이 달라지면 예측 오류가 발생할 수 있습니다.

In [ ]:

schema_df = pd.DataFrame({
    "column_name": bank_df.columns,
    "dtype": bank_df.dtypes.astype(str).values,
    "non_null_count": bank_df.notnull().sum().values,
    "null_count": bank_df.isnull().sum().values
})

display(schema_df)

## 5. 타깃 변수 분리 및 클래스 분포 확인

### 비즈니스 목적과 기대 효과
이탈 여부(`churn`) 분포를 확인하면, 모델 평가 기준과 데이터 분할 전략을 설계하는 데 필요한 기초 정보를 확보할 수 있습니다.

### 분석 포인트
- 이탈 고객 비율이 낮다면 불균형 데이터 문제를 고려해야 합니다.
- 불균형 데이터에서는 정확도보다 Recall, PR-AUC가 더 중요한 지표가 될 수 있습니다.

In [ ]:

target_col = "churn"

# 타깃과 입력 변수를 분리해 파이프라인 혼선을 방지합니다.
churn_df = bank_df[target_col]
feature_df = bank_df.drop(columns=[target_col])

target_summary_df = (
    churn_df.value_counts(dropna=False)
    .rename_axis("churn")
    .reset_index(name="count")
)
target_summary_df["ratio"] = target_summary_df["count"] / target_summary_df["count"].sum()

display(target_summary_df)

## 6. 입력 변수 목록 확인

### 비즈니스 목적과 기대 효과
어떤 변수가 모델 입력으로 사용되는지 명확히 정의하면, 팀 내 커뮤니케이션과 추후 피처 엔지니어링 범위 설정이 쉬워집니다.

### 분석 포인트
- 현재 입력 변수에는 인구통계, 카드 정보, 거래 정보가 함께 포함되어 있습니다.
- 범주형 코드형 변수와 연속형 변수의 해석 방식을 구분해야 합니다.

In [ ]:

feature_list_df = pd.DataFrame({"feature_name": feature_df.columns})
display(feature_list_df)

## 7. 학습/평가 데이터 분리

### 비즈니스 목적과 기대 효과
모델이 본 적 없는 데이터에서 얼마나 잘 동작하는지 확인하려면 학습용과 평가용 데이터를 분리해야 합니다.

### 분석 포인트
- `stratify`를 사용하면 불균형 데이터에서도 train/test 간 클래스 비율 차이를 줄일 수 있습니다.
- 실무에서는 재현 가능한 비교를 위해 `random_state`를 고정하는 것이 중요합니다.

In [ ]:

from sklearn.model_selection import train_test_split

train_X, test_X, train_y, test_y = train_test_split(
    feature_df,
    churn_df,
    test_size=0.2,
    stratify=churn_df,
    random_state=42
)

display(train_X.head())

## 8. 학습 데이터 기술통계 확인

### 비즈니스 목적과 기대 효과
학습 데이터의 수치 분포를 점검하면, 이상치 가능성, 변수 스케일 차이, 데이터 품질 문제를 조기에 발견할 수 있습니다.

### 분석 포인트
- `credit_limit`, `transaction_amount` 등은 스케일 차이가 큰 변수입니다.
- 일부 비율형 변수는 0~1 범위에 가까워 다른 수치형 변수와 분포 특성이 다릅니다.
- 트리 계열 모델은 스케일 영향이 비교적 작지만, 선형 모델·거리 기반 모델은 민감할 수 있습니다.

In [ ]:

train_describe_df = train_X.describe().T
display(train_describe_df)

## 9. 학습/평가 데이터 분리 결과 검증

### 비즈니스 목적과 기대 효과
분리 후 행 수와 클래스 비율이 적절한지 확인하면 모델 성능 비교 결과의 신뢰도를 높일 수 있습니다.

### 분석 포인트
- Train/Test 클래스 비율이 유사하면 분리 품질이 양호하다고 볼 수 있습니다.
- 이후 성능 비교 시 데이터 분리의 편향이 아니라 모델 차이로 해석하기 쉬워집니다.

In [ ]:

def make_split_report(X_train, X_test, y_train, y_test) -> pd.DataFrame:
    report_rows = [
        ["X_train shape", X_train.shape],
        ["X_test shape", X_test.shape],
        ["y_train shape", y_train.shape],
        ["y_test shape", y_test.shape],
        ["train class 0 ratio", (y_train == 0).mean()],
        ["train class 1 ratio", (y_train == 1).mean()],
        ["test class 0 ratio", (y_test == 0).mean()],
        ["test class 1 ratio", (y_test == 1).mean()],
    ]
    return pd.DataFrame(report_rows, columns=["item", "value"])

split_report_df = make_split_report(train_X, test_X, train_y, test_y)
display(split_report_df)

train_dist_df = y_train.value_counts(normalize=True).sort_index().rename("train_ratio").reset_index()
test_dist_df = y_test.value_counts(normalize=True).sort_index().rename("test_ratio").reset_index()
dist_compare_df = train_dist_df.merge(test_dist_df, on="churn")
display(dist_compare_df)

## 10. 주요 수치형 변수 상관관계 확인

### 비즈니스 목적과 기대 효과
상관관계 분석은 중복 정보가 많은 변수 조합이나 비즈니스적으로 함께 움직이는 지표를 파악하는 데 유용합니다.

### 분석 포인트
- 상관관계는 인과관계를 의미하지는 않습니다.
- 강한 상관관계가 있는 변수 쌍은 해석이나 피처 축소 시 참고할 수 있습니다.
- 모델 성능보다 설명 가능성 향상에 더 직접적으로 기여하는 단계입니다.

In [ ]:

numeric_cols = feature_df.select_dtypes(include=[np.number]).columns.tolist()
corr_df = train_X[numeric_cols].corr()

display(corr_df.round(3))

## 11. 상관관계 히트맵 시각화

### 비즈니스 목적과 기대 효과
표 형태 상관계수만으로는 패턴을 파악하기 어렵기 때문에, 히트맵으로 구조를 시각화하면 변수 그룹별 관계를 빠르게 이해할 수 있습니다.

### 분석 포인트
- `credit_limit`, `available_credit`, `revolving_balance`, `utilization_ratio`는 함께 해석할 필요가 있습니다.
- 거래량 관련 변수와 변화율 변수도 별도 묶음으로 볼 수 있습니다.

In [ ]:

fig, ax = plt.subplots(figsize=(14, 10))
im = ax.imshow(corr_df, aspect="auto")

ax.set_title("수치형 변수 상관관계 히트맵")
ax.set_xlabel("변수")
ax.set_ylabel("변수")
ax.set_xticks(range(len(corr_df.columns)))
ax.set_xticklabels(corr_df.columns, rotation=90)
ax.set_yticks(range(len(corr_df.index)))
ax.set_yticklabels(corr_df.index)

cbar = plt.colorbar(im, ax=ax)
cbar.set_label("상관계수")

plt.tight_layout()
plt.show()

## 12. 타깃 기준 평균 비교

### 비즈니스 목적과 기대 효과
이탈 고객과 비이탈 고객 간 평균 차이를 비교하면, 어떤 변수가 이탈 신호와 연관되는지 초기에 탐색할 수 있습니다.

### 분석 포인트
- 거래 횟수, 거래 금액, 활동성 관련 변수는 이탈 예측의 핵심 후보일 가능성이 높습니다.
- 평균 차이가 크더라도 분산이 크면 실제 예측력은 제한적일 수 있습니다.

In [ ]:

train_with_target_df = train_X.copy()
train_with_target_df["churn"] = train_y.values

group_mean_df = train_with_target_df.groupby("churn").mean(numeric_only=True).T
group_mean_df.columns = ["non_churn_mean", "churn_mean"]
group_mean_df["abs_diff"] = (group_mean_df["churn_mean"] - group_mean_df["non_churn_mean"]).abs()
group_mean_df = group_mean_df.sort_values("abs_diff", ascending=False)

display(group_mean_df.head(15))

## 13. 주요 변수별 분포 시각화

### 비즈니스 목적과 기대 효과
평균만으로는 분포 차이를 충분히 설명하기 어렵기 때문에, 핵심 변수 몇 개를 대상으로 클래스별 분포를 직접 시각화합니다.

### 분석 포인트
- 거래 관련 변수의 분포 차이가 뚜렷하면 이탈 탐지에 실질적으로 기여할 가능성이 큽니다.
- 분포가 크게 겹치는 변수는 단독 변수로는 설명력이 낮을 수 있습니다.

In [ ]:

key_features = ["transaction_count", "transaction_amount", "revolving_balance", "credit_limit"]

for col in key_features:
    fig, ax = plt.subplots(figsize=(9, 5))

    non_churn_values = train_with_target_df.loc[train_with_target_df["churn"] == 0, col].dropna()
    churn_values = train_with_target_df.loc[train_with_target_df["churn"] == 1, col].dropna()

    ax.hist(non_churn_values, bins=30, alpha=0.6, label="Non-Churn")
    ax.hist(churn_values, bins=30, alpha=0.6, label="Churn")

    ax.set_title(f"{col} 분포 비교")
    ax.set_xlabel(col)
    ax.set_ylabel("빈도")
    ax.legend()

    plt.tight_layout()
    plt.show()

## 14. 통계 검정 준비

### 비즈니스 목적과 기대 효과
시각적 차이가 실제로 통계적으로 유의한지 확인하면, 변수 선택과 해석의 객관성을 높일 수 있습니다.

### 분석 포인트
- 여기서는 두 집단 평균 차이를 빠르게 검토하기 위해 Welch's t-test를 사용합니다.
- 분포 비정규성이나 이상치가 심하면 비모수 검정을 추가 검토할 수 있습니다.

In [ ]:

from scipy.stats import ttest_ind

## 15. 주요 변수 통계 검정

### 비즈니스 목적과 기대 효과
이탈/비이탈 집단 간 차이가 우연인지 아닌지를 확인해 설명 변수 후보의 신뢰도를 높입니다.

### 분석 포인트
- p-value가 매우 작다면 두 집단 평균 차이가 우연일 가능성이 낮다고 해석할 수 있습니다.
- 다만 표본 수가 크면 작은 차이도 유의하게 나올 수 있으므로 효과 크기와 함께 해석해야 합니다.

In [ ]:

stat_results = []

for col in key_features:
    non_churn_values = train_with_target_df.loc[train_with_target_df["churn"] == 0, col].dropna()
    churn_values = train_with_target_df.loc[train_with_target_df["churn"] == 1, col].dropna()

    stat, pvalue = ttest_ind(non_churn_values, churn_values, equal_var=False)

    stat_results.append({
        "feature": col,
        "non_churn_mean": non_churn_values.mean(),
        "churn_mean": churn_values.mean(),
        "t_statistic": stat,
        "p_value": pvalue
    })

stat_results_df = pd.DataFrame(stat_results).sort_values("p_value")
display(stat_results_df)

## 16. 평가 함수 정의

### 비즈니스 목적과 기대 효과
모델별 성능 비교 기준을 일관되게 맞추면, 실험 결과를 신뢰성 있게 비교하고 최종 모델을 선정할 수 있습니다.

### 분석 포인트
- 불균형 이진분류에서는 Accuracy 외에 Recall, Precision, F1, ROC-AUC, PR-AUC를 함께 봐야 합니다.
- PR-AUC는 양성 클래스가 적을 때 더 실무적인 지표가 될 수 있습니다.

In [ ]:

from sklearn.metrics import (
    accuracy_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    precision_score
)

def evaluate_binary_classifier(y_true, y_pred, y_proba, dataset_name="Dataset") -> pd.DataFrame:
    y_proba = np.asarray(y_proba)
    if y_proba.ndim == 2 and y_proba.shape[1] == 2:
        y_score = y_proba[:, 1]
    else:
        y_score = y_proba

    result = pd.DataFrame([{
        "dataset": dataset_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "recall": recall_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred),
        "f1_score": f1_score(y_true, y_pred),
        "roc_auc": roc_auc_score(y_true, y_score),
        "pr_auc": average_precision_score(y_true, y_score)
    }])
    return result

## 17. 기준 모델 1차 학습: HistGradientBoostingClassifier

### 비즈니스 목적과 기대 효과
기본 성능이 우수한 트리 기반 부스팅 모델을 먼저 학습해 현재 데이터에서 달성 가능한 기준 성능선을 설정합니다.

### 분석 포인트
- `HistGradientBoostingClassifier`는 수치형 데이터에서 효율적이고 성능이 좋은 편입니다.
- `early_stopping`을 사용하면 불필요한 반복 학습을 줄여 과적합과 계산 비용을 동시에 관리할 수 있습니다.

In [ ]:

from sklearn.ensemble import HistGradientBoostingClassifier

baseline_hgb_clf = HistGradientBoostingClassifier(
    random_state=42,
    max_iter=5000,
    validation_fraction=0.2,
    early_stopping=True,
    n_iter_no_change=10
)

baseline_hgb_clf.fit(train_X, train_y)

baseline_train_pred = baseline_hgb_clf.predict(train_X)
baseline_test_pred = baseline_hgb_clf.predict(test_X)

baseline_train_proba = baseline_hgb_clf.predict_proba(train_X)
baseline_test_proba = baseline_hgb_clf.predict_proba(test_X)

## 18. 기준 모델 성능 평가

### 비즈니스 목적과 기대 효과
기준 모델의 학습/테스트 성능을 함께 보면 일반화 성능과 과적합 여부를 동시에 판단할 수 있습니다.

### 분석 포인트
- Train 성능이 지나치게 높고 Test 성능과 차이가 크면 과적합 가능성을 의심할 수 있습니다.
- 불균형 환경에서는 Test 기준 Recall과 PR-AUC를 특히 주의 깊게 봐야 합니다.

In [ ]:

baseline_train_metrics_df = evaluate_binary_classifier(
    y_true=train_y,
    y_pred=baseline_train_pred,
    y_proba=baseline_train_proba,
    dataset_name="Baseline Train"
)

baseline_test_metrics_df = evaluate_binary_classifier(
    y_true=test_y,
    y_pred=baseline_test_pred,
    y_proba=baseline_test_proba,
    dataset_name="Baseline Test"
)

baseline_metrics_df = pd.concat([baseline_train_metrics_df, baseline_test_metrics_df], ignore_index=True)
display(baseline_metrics_df)

## 19. 하이퍼파라미터 탐색 설계

### 비즈니스 목적과 기대 효과
성능 향상을 위해 반복 횟수, 조기 종료 기준, 검증 데이터 비율을 조정하며 모델이 데이터 특성에 더 잘 맞도록 탐색합니다.

### 분석 포인트
- `max_iter`는 모델 복잡도와 직접적으로 연결됩니다.
- `validation_fraction`은 조기 종료 기준의 안정성에 영향을 줍니다.
- 너무 촘촘한 GridSearch는 시간이 오래 걸리므로 단계적으로 좁혀가는 전략이 실무적입니다.

In [ ]:

from sklearn.model_selection import GridSearchCV

search_model = HistGradientBoostingClassifier(
    random_state=42,
    early_stopping=True
)

param_grid_stage1 = {
    "max_iter": [800, 1000, 1200],
    "validation_fraction": [0.1, 0.2, 0.3],
    "n_iter_no_change": [15, 20, 25],
}

grid_search_stage1 = GridSearchCV(
    estimator=search_model,
    param_grid=param_grid_stage1,
    cv=5,
    scoring="roc_auc",
    n_jobs=-1
)

grid_search_stage1.fit(train_X, train_y)

stage1_result_df = pd.DataFrame([{
    "stage": "stage_1",
    "best_params": str(grid_search_stage1.best_params_),
    "best_score": grid_search_stage1.best_score_
}])

display(stage1_result_df)

## 20. 2차 하이퍼파라미터 탐색

### 비즈니스 목적과 기대 효과
1차 탐색 결과를 기반으로 범위를 좁혀 재탐색하면, 계산 비용을 크게 늘리지 않고 더 안정적인 최적 구간을 찾을 수 있습니다.

### 분석 포인트
- 반복 탐색 결과가 일관되게 특정 구간으로 수렴하는지 확인하는 것이 중요합니다.
- 최고 점수 자체보다 **점수 변화 폭이 큰지 작은지**도 함께 해석해야 합니다.

In [ ]:

search_model_stage2 = HistGradientBoostingClassifier(
    random_state=42,
    early_stopping=True,
    n_iter_no_change=20
)

param_grid_stage2 = {
    "max_iter": [200, 500, 800],
    "validation_fraction": [0.05, 0.1, 0.15],
}

grid_search_stage2 = GridSearchCV(
    estimator=search_model_stage2,
    param_grid=param_grid_stage2,
    cv=5,
    scoring="roc_auc",
    n_jobs=-1
)

grid_search_stage2.fit(train_X, train_y)

stage2_result_df = pd.DataFrame([{
    "stage": "stage_2",
    "best_params": str(grid_search_stage2.best_params_),
    "best_score": grid_search_stage2.best_score_
}])

display(stage2_result_df)

## 21. 3차 하이퍼파라미터 탐색

### 비즈니스 목적과 기대 효과
최적 구간 근처를 더 세밀하게 확인해 운영 관점에서 성능과 학습 비용의 균형이 좋은 설정을 선택합니다.

### 분석 포인트
- 작은 성능 차이만 존재한다면 더 단순하고 빠른 설정이 실무적으로 유리할 수 있습니다.
- 반복 횟수가 너무 많을수록 학습 시간은 증가하지만 성능 이득은 제한적일 수 있습니다.

In [ ]:

search_model_stage3 = HistGradientBoostingClassifier(
    random_state=42,
    early_stopping=True,
    n_iter_no_change=20
)

param_grid_stage3 = {
    "max_iter": [100, 200, 300],
    "validation_fraction": [0.03, 0.05, 0.08],
}

grid_search_stage3 = GridSearchCV(
    estimator=search_model_stage3,
    param_grid=param_grid_stage3,
    cv=5,
    scoring="roc_auc",
    n_jobs=-1
)

grid_search_stage3.fit(train_X, train_y)

stage3_result_df = pd.DataFrame([{
    "stage": "stage_3",
    "best_params": str(grid_search_stage3.best_params_),
    "best_score": grid_search_stage3.best_score_
}])

display(stage3_result_df)

## 22. 4차 하이퍼파라미터 탐색

### 비즈니스 목적과 기대 효과
가장 유망한 `max_iter` 범위를 한 번 더 좁혀 최종 후보 모델 설정을 결정합니다.

### 분석 포인트
- 탐색 범위를 줄이는 이유는 계산 효율을 높이면서도 최적점을 놓치지 않기 위해서입니다.
- 점수 차이가 미미하다면 운영 복잡도가 낮은 설정을 우선 고려할 수 있습니다.

In [ ]:

search_model_stage4 = HistGradientBoostingClassifier(
    random_state=42,
    early_stopping=True,
    n_iter_no_change=20,
    validation_fraction=0.05
)

param_grid_stage4 = {
    "max_iter": [50, 100, 150]
}

grid_search_stage4 = GridSearchCV(
    estimator=search_model_stage4,
    param_grid=param_grid_stage4,
    cv=5,
    scoring="roc_auc",
    n_jobs=-1
)

grid_search_stage4.fit(train_X, train_y)

stage4_result_df = pd.DataFrame([{
    "stage": "stage_4",
    "best_params": str(grid_search_stage4.best_params_),
    "best_score": grid_search_stage4.best_score_
}])

display(stage4_result_df)

## 23. 5차 하이퍼파라미터 탐색

### 비즈니스 목적과 기대 효과
최종적으로 가장 유망한 반복 횟수 인근을 비교해 배포 후보 모델의 설정을 확정합니다.

### 분석 포인트
- 단계적 탐색 결과가 `max_iter=150` 근처로 수렴하는지 확인합니다.
- 최고 점수뿐 아니라 이전 단계 대비 개선 폭이 얼마나 작은지도 함께 봅니다.

In [ ]:

search_model_stage5 = HistGradientBoostingClassifier(
    random_state=42,
    early_stopping=True,
    n_iter_no_change=20,
    validation_fraction=0.05
)

param_grid_stage5 = {
    "max_iter": [120, 150, 180]
}

grid_search_stage5 = GridSearchCV(
    estimator=search_model_stage5,
    param_grid=param_grid_stage5,
    cv=5,
    scoring="roc_auc",
    n_jobs=-1
)

grid_search_stage5.fit(train_X, train_y)

stage5_result_df = pd.DataFrame([{
    "stage": "stage_5",
    "best_params": str(grid_search_stage5.best_params_),
    "best_score": grid_search_stage5.best_score_
}])

display(stage5_result_df)

## 24. 하이퍼파라미터 탐색 결과 종합

### 비즈니스 목적과 기대 효과
탐색 결과를 한 표로 모아보면 최적 설정이 어느 방향으로 수렴했는지 팀 단위로 빠르게 공유할 수 있습니다.

### 분석 포인트
- 반복 탐색 결과가 안정적으로 유사한 설정을 가리키면 신뢰도가 높아집니다.
- 최고 점수 차이가 매우 작다면, 성능보다 학습 시간과 단순성을 더 우선할 수 있습니다.

In [ ]:

tuning_summary_df = pd.concat(
    [stage1_result_df, stage2_result_df, stage3_result_df, stage4_result_df, stage5_result_df],
    ignore_index=True
)

display(tuning_summary_df)

## 25. 최종 모델 정의 및 학습

### 비즈니스 목적과 기대 효과
탐색 결과를 반영한 최종 모델을 학습해 운영 적용 후보의 실제 성능을 평가합니다.

### 분석 포인트
- 최종 모델은 GridSearch에서 얻은 방향을 반영하되, 복잡도와 재현 가능성을 함께 고려해 설정합니다.
- 제공 코드에는 `model`을 생성해 놓고 `hgb_clf`를 다시 사용하는 실수가 있었으므로, 여기서는 객체를 명확히 분리해 유지보수성을 높입니다.

In [ ]:

final_hgb_clf = HistGradientBoostingClassifier(
    max_iter=150,
    random_state=42,
    early_stopping=True,
    n_iter_no_change=20,
    validation_fraction=0.05
)

final_hgb_clf.fit(train_X, train_y)

final_train_pred = final_hgb_clf.predict(train_X)
final_test_pred = final_hgb_clf.predict(test_X)

final_train_proba = final_hgb_clf.predict_proba(train_X)
final_test_proba = final_hgb_clf.predict_proba(test_X)

## 26. 최종 모델 성능 평가

### 비즈니스 목적과 기대 효과
최종 모델의 학습/테스트 성능을 확인해 실제 운영 반영 후보로 적합한지 판단합니다.

### 분석 포인트
- Test ROC-AUC와 PR-AUC가 높다면 고객 이탈 탐지 성능이 양호하다고 볼 수 있습니다.
- Recall이 충분히 높다면 실제 이탈 고객을 놓치는 비율을 줄일 수 있습니다.
- Precision과 Recall의 균형은 캠페인 비용 구조에 따라 다르게 최적화할 수 있습니다.

In [ ]:

final_train_metrics_df = evaluate_binary_classifier(
    y_true=train_y,
    y_pred=final_train_pred,
    y_proba=final_train_proba,
    dataset_name="Final Train"
)

final_test_metrics_df = evaluate_binary_classifier(
    y_true=test_y,
    y_pred=final_test_pred,
    y_proba=final_test_proba,
    dataset_name="Final Test"
)

final_metrics_df = pd.concat([final_train_metrics_df, final_test_metrics_df], ignore_index=True)
display(final_metrics_df)

## 27. 기준 모델과 최종 모델 비교

### 비즈니스 목적과 기대 효과
튜닝이 실제로 의미 있는 개선을 만들었는지 확인해야 추가 탐색 비용이 정당했는지 판단할 수 있습니다.

### 분석 포인트
- 성능 향상 폭이 작다면 더 단순한 모델을 유지하는 것이 운영상 유리할 수 있습니다.
- 반대로 Recall이나 PR-AUC가 개선되었다면 불균형 분류 실무에서 충분한 가치가 있습니다.

In [ ]:

model_compare_df = pd.concat(
    [
        baseline_test_metrics_df.assign(model_name="baseline_hgb"),
        final_test_metrics_df.assign(model_name="final_hgb")
    ],
    ignore_index=True
)[["model_name", "dataset", "accuracy", "recall", "precision", "f1_score", "roc_auc", "pr_auc"]]

display(model_compare_df)

## 28. 변수 중요도 확인

### 비즈니스 목적과 기대 효과
중요 변수를 확인하면 이탈 방지 전략 수립과 후속 피처 엔지니어링 방향 설정에 도움이 됩니다.

### 분석 포인트
- 거래 횟수, 거래 금액, 활동성 변수는 중요도가 높게 나올 가능성이 큽니다.
- 중요도는 예측 기여도일 뿐, 직접적인 원인이라고 해석하면 안 됩니다.

In [ ]:

feature_importance_df = pd.DataFrame({
    "feature": train_X.columns,
    "importance": final_hgb_clf.feature_importances_
}).sort_values("importance", ascending=False)

display(feature_importance_df.head(15))

## 29. 변수 중요도 시각화

### 비즈니스 목적과 기대 효과
상위 중요 변수를 시각화하면 비기술 직군도 모델이 어떤 신호를 주로 활용하는지 빠르게 이해할 수 있습니다.

### 분석 포인트
- 중요도 상위 변수는 이탈 방지 캠페인 타겟팅 로직 설계 시 우선 검토 대상입니다.
- 다만 단일 지표로 정책을 만들기보다 복합적으로 활용해야 합니다.

In [ ]:

top_n = 10
top_importance_df = feature_importance_df.head(top_n).sort_values("importance", ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top_importance_df["feature"], top_importance_df["importance"])

ax.set_title("최종 모델 상위 변수 중요도")
ax.set_xlabel("Importance")
ax.set_ylabel("Feature")

plt.tight_layout()
plt.show()

## 30. 최종 해석 요약

### 비즈니스 목적과 기대 효과
분석 결과를 한 번 더 문장형으로 정리하면 회의 자료나 보고용으로 바로 전환하기 쉽습니다.

### 분석 포인트
- 불균형 데이터를 고려할 때 정확도보다 Recall/PR-AUC 중심 해석이 중요합니다.
- 모델 튜닝 결과는 수렴 구간을 찾는 데 의미가 있으며, 큰 폭 개선보다는 안정적 최적화에 가깝습니다.
- 중요 변수는 향후 CRM 전략과 피처 엔지니어링 후보로 활용할 수 있습니다.

In [ ]:

summary_lines = [
    "[분석 요약]",
    "1. 데이터는 churn=1 비율이 낮은 불균형 이진분류 구조입니다.",
    "2. 최종 테스트 성능은 아래와 같습니다.",
    f"   - Accuracy : {final_test_metrics_df.loc[0, 'accuracy']:.4f}",
    f"   - Recall   : {final_test_metrics_df.loc[0, 'recall']:.4f}",
    f"   - Precision: {final_test_metrics_df.loc[0, 'precision']:.4f}",
    f"   - F1-score : {final_test_metrics_df.loc[0, 'f1_score']:.4f}",
    f"   - ROC-AUC  : {final_test_metrics_df.loc[0, 'roc_auc']:.4f}",
    f"   - PR-AUC   : {final_test_metrics_df.loc[0, 'pr_auc']:.4f}",
    "3. 하이퍼파라미터 탐색 결과는 max_iter=150, validation_fraction=0.05 부근으로 수렴했습니다.",
    "4. 상위 중요 변수는 이탈 징후를 설명하는 핵심 후보로, 향후 고객 관리 정책 설계에 활용 가능합니다."
]

print("\n".join(summary_lines))

## 31. 결과 저장

### 비즈니스 목적과 기대 효과
최종 성능표와 중요도표를 파일로 저장해 두면 회의 자료, 배포 문서, 실험 이력 관리에 재사용할 수 있습니다.

### 분석 포인트
- 저장 경로는 환경마다 다를 수 있으므로 상대경로/절대경로 정책을 팀 내에서 통일하는 것이 좋습니다.
- 운영 서버에서는 권한 문제로 저장 실패가 날 수 있으므로 예외 처리가 필요합니다.

In [ ]:

from pathlib import Path

output_dir = Path("./analysis_outputs")
output_dir.mkdir(parents=True, exist_ok=True)

baseline_metrics_df.to_csv(output_dir / "baseline_metrics.csv", index=False, encoding="utf-8-sig")
final_metrics_df.to_csv(output_dir / "final_metrics.csv", index=False, encoding="utf-8-sig")
tuning_summary_df.to_csv(output_dir / "tuning_summary.csv", index=False, encoding="utf-8-sig")
feature_importance_df.to_csv(output_dir / "feature_importance.csv", index=False, encoding="utf-8-sig")

print(f"저장 완료: {output_dir.resolve()}")